# ThreatLens — Task 4: Data Collection, EDA & Preprocessing

**Dataset:** CICIDS2017 cleaned Parquet version (8 files)  
**Project:** ThreatLens — Intelligent Cybersecurity Decision Support System

This notebook:
1. Loads all 8 Parquet files from the uploaded ZIP.
2. Combines them into one dataset.
3. Examines structure, data types and statistics.
4. Checks missing/infinite values and duplicates.
5. Detects outliers using IQR.
6. Visualizes class distributions, feature distributions and correlations.
7. Cleans invalid values and duplicates.
8. Creates a cleaned CSV for the next project phases.
9. Demonstrates scaling without data leakage using a train/test split.


## 1. Why this dataset is suitable for ThreatLens

The uploaded archive contains the eight CICIDS2017 Parquet files:

- Benign-Monday-no-metadata.parquet
- Botnet-Friday-no-metadata.parquet
- Bruteforce-Tuesday-no-metadata.parquet
- DDoS-Friday-no-metadata.parquet
- DoS-Wednesday-no-metadata.parquet
- Infiltration-Thursday-no-metadata.parquet
- Portscan-Friday-no-metadata.parquet
- WebAttacks-Thursday-no-metadata.parquet

The dataset contains 78 columns: 77 network-flow features and the `Label` target. The labels include benign traffic and multiple attack types, making it appropriate for intrusion/threat detection. 


In [ ]:
# 2. Install/read dependencies in Google Colab
# Colab normally has pandas, numpy, matplotlib and seaborn.
# PyArrow is required to read Parquet files.

!pip -q install pyarrow scikit-learn

import os
import zipfile
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully.")


In [ ]:
# 3. Upload the dataset ZIP
from google.colab import files

uploaded = files.upload()

zip_name = next(iter(uploaded.keys()))
print("Uploaded:", zip_name)


In [ ]:
# 4. Extract the ZIP
DATA_DIR = "/content/ThreatLens_CICIDS2017"

os.makedirs(DATA_DIR, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(DATA_DIR)

parquet_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.parquet")))

print("Parquet files found:", len(parquet_files))
for f in parquet_files:
    print(os.path.basename(f))


## 2. Load all files

Do not treat the files as unrelated datasets. They are class/day partitions of the same CICIDS2017 flow dataset, so they are combined for the ThreatLens analysis.


In [ ]:
# 5. Load each Parquet file and inspect its size
file_info = []

for path in parquet_files:
    temp = pd.read_parquet(path)
    file_info.append({
        "File": os.path.basename(path),
        "Rows": len(temp),
        "Columns": len(temp.columns),
        "Label_Values": temp["Label"].nunique() if "Label" in temp.columns else None
    })
    print(f"{os.path.basename(path):45s} -> {temp.shape}")

file_info_df = pd.DataFrame(file_info)
display(file_info_df)


In [ ]:
# 6. Combine all eight files
dfs = [pd.read_parquet(path) for path in parquet_files]

df = pd.concat(dfs, ignore_index=True)

print("Combined shape:", df.shape)
display(df.head())


In [ ]:
# 7. Dataset structure
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.value_counts())

print("\nDataset info:")
df.info()


### Expected dataset size

For this exact archive, the combined dataset should contain approximately **2,313,810 rows and 78 columns** before duplicate removal.

The dataset contains **77 numerical flow features + `Label`**. The features include destination port, flow duration, packet counts, packet lengths, inter-arrival times, TCP flag counts, window sizes, active/idle statistics and related flow measures.


In [ ]:
# 8. Numerical and categorical feature identification
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print("Number of numerical columns:", len(numeric_cols))
print("Number of categorical columns:", len(categorical_cols))

print("\nCategorical columns:")
print(categorical_cols)

print("\nFirst 20 numerical columns:")
print(numeric_cols[:20])


In [ ]:
# 9. Summary statistics
display(df.describe())

print("\nDetailed statistics:")
display(df[numeric_cols].describe().T)


## 3. Missing-value and invalid-value analysis

Missing values and infinite values are checked before modeling. The cleaned Parquet version of this dataset is expected to have no missing records, but the notebook verifies this rather than assuming it.


In [ ]:
# 10. Missing values
missing = df.isnull().sum()
missing_table = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": (missing / len(df)) * 100
}).sort_values("Missing_Count", ascending=False)

display(missing_table[missing_table["Missing_Count"] > 0])

print("TOTAL MISSING VALUES:", int(df.isnull().sum().sum()))


In [ ]:
# 11. Infinite values
inf_counts = pd.Series(
    np.isinf(df[numeric_cols]).sum(),
    index=numeric_cols
).sort_values(ascending=False)

display(inf_counts[inf_counts > 0])

print("TOTAL INFINITE VALUES:", int(inf_counts.sum()))


In [ ]:
# 12. Handle invalid values safely
# Replace +/- infinity with NaN.
df = df.replace([np.inf, -np.inf], np.nan)

# Numeric missing values -> median
for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

# Categorical missing values -> mode
for col in categorical_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after treatment:", int(df.isnull().sum().sum()))


## 4. Duplicate analysis

Duplicate flow records can bias later modeling, so exact duplicate rows are identified and removed.


In [ ]:
# 13. Duplicate rows
duplicate_count = int(df.duplicated().sum())

print("Duplicate rows found:", duplicate_count)
print("Shape before duplicate removal:", df.shape)

df = df.drop_duplicates().reset_index(drop=True)

print("Shape after duplicate removal:", df.shape)
print("Remaining duplicates:", int(df.duplicated().sum()))


### Expected duplicate result

For this exact CICIDS2017 archive, the combined data contains **82,004 duplicate rows** and the cleaned dataset contains **2,231,806 rows and 78 columns** after duplicate removal.


In [ ]:
# 14. Final validation after basic cleaning
print("Shape:", df.shape)
print("Missing values:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Infinite values:", int(np.isinf(df[numeric_cols]).sum().sum()))


## 5. Target/attack analysis

`Label` is the target variable. We retain the original multi-class label because ThreatLens is intended to explain different threat categories, not only benign-vs-attack traffic.


In [ ]:
# 15. Multi-class attack distribution
label_counts = df["Label"].value_counts()

print("Number of classes:", df["Label"].nunique())
display(label_counts.to_frame("Count"))

label_percent = (label_counts / len(df) * 100).round(4)
display(label_percent.to_frame("Percentage"))


In [ ]:
# 16. Attack distribution bar chart
plt.figure(figsize=(13, 6))
label_counts.sort_values(ascending=False).plot(kind="bar")
plt.title("ThreatLens — Distribution of Security Event / Attack Classes")
plt.xlabel("Security Event / Attack Type")
plt.ylabel("Number of Records")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Known class distribution for this cleaned version

The expected cleaned counts are:

| Label | Count |
|---|---:|
| Benign | 1,895,314 |
| DoS Hulk | 172,846 |
| DDoS | 128,014 |
| DoS GoldenEye | 10,286 |
| FTP-Patator | 5,931 |
| DoS slowloris | 5,385 |
| DoS Slowhttptest | 5,228 |
| SSH-Patator | 3,219 |
| PortScan | 1,956 |
| Web Attack Brute Force | 1,470 |
| Bot | 1,437 |
| Web Attack XSS | 652 |
| Infiltration | 36 |
| Web Attack Sql Injection | 21 |
| Heartbleed | 11 |

This shows a strong class imbalance, with benign traffic dominating the dataset.


## 6. Binary threat view

For some later models, ThreatLens may use:
- `0` = Benign
- `1` = Attack

But the original multi-class `Label` should be preserved for explanation and multi-class classification.


In [ ]:
# 17. Create a binary threat label without deleting the original Label
df["Threat_Binary"] = np.where(
    df["Label"].str.strip().str.lower().eq("benign"),
    0,
    1
)

binary_counts = df["Threat_Binary"].map({
    0: "Benign",
    1: "Attack"
}).value_counts()

display(binary_counts.to_frame("Count"))

plt.figure(figsize=(7, 5))
binary_counts.plot(kind="bar")
plt.title("ThreatLens — Benign vs Attack Distribution")
plt.xlabel("Traffic Category")
plt.ylabel("Number of Records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. Outlier detection using IQR

Cybersecurity traffic naturally contains extreme values. A large flow or packet count may be a real attack signal. Therefore, Task 4 **detects** outliers but does not blindly delete them.

The IQR method is used on selected high-value numeric features.


In [ ]:
# 18. Select representative features for outlier analysis
candidate_features = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Packet Length Mean",
    "Packet Length Variance"
]

outlier_features = [c for c in candidate_features if c in df.columns]

print("Features selected for outlier analysis:")
print(outlier_features)


In [ ]:
# 19. IQR outlier table
outlier_results = []

for col in outlier_features:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = int(((df[col] < lower) | (df[col] > upper)).sum())

    outlier_results.append({
        "Feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": count,
        "Outlier_Percentage": round(count / len(df) * 100, 4)
    })

outlier_table = pd.DataFrame(outlier_results)
display(outlier_table)


In [ ]:
# 20. Boxplots for representative features
for col in outlier_features[:6]:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=df[col])
    plt.title(f"ThreatLens Outlier Detection — {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()


### Outlier treatment decision

Do **not** remove every IQR outlier in this dataset. Extreme traffic values can correspond to genuine DoS/DDoS or other malicious behavior. Only impossible/invalid values are corrected or removed. This preserves cybersecurity evidence for later detection models.


## 8. Feature distribution visualization

The full dataset has many features, so representative features are visualized instead of generating dozens of unreadable plots.


In [ ]:
# 21. Histograms of representative numeric features
hist_features = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Packet Length Mean",
    "Flow Bytes/s",
    "Flow Packets/s"
]
hist_features = [c for c in hist_features if c in df.columns]

# Sample for visualization to keep plots fast
plot_sample = df.sample(
    n=min(100000, len(df)),
    random_state=42
)

for col in hist_features:
    plt.figure(figsize=(9, 4))
    sns.histplot(plot_sample[col], bins=50, kde=False)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


In [ ]:
# 22. Feature vs attack class
feature_for_class_plot = "Flow Duration"

plt.figure(figsize=(13, 6))
sns.boxplot(
    data=plot_sample,
    x="Label",
    y=feature_for_class_plot
)
plt.title(f"{feature_for_class_plot} Across Security Event Classes")
plt.xlabel("Security Event / Attack Type")
plt.ylabel(feature_for_class_plot)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9. Correlation analysis

Correlation is calculated only on numeric features. Because there are 77 features, a full heatmap can be large; the notebook also provides a focused heatmap for representative features.


In [ ]:
# 23. Correlation matrix for representative features
corr_features = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Packet Length Mean",
    "Packet Length Std",
    "Packet Length Variance",
    "SYN Flag Count",
    "ACK Flag Count",
    "Average Packet Size"
]
corr_features = [c for c in corr_features if c in df.columns]

corr = df[corr_features].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", center=0)
plt.title("ThreatLens — Correlation Heatmap of Selected Network Features")
plt.tight_layout()
plt.show()


In [ ]:
# 24. Highest absolute correlations among selected features
corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .abs()
        .sort_values(ascending=False)
)

display(corr_pairs.head(15).to_frame("Absolute_Correlation"))


## 10. Prepare the cleaned dataset

Two outputs are created:

**A. `ThreatLens_Cleaned.csv`**  
Preserves the original multi-class `Label` and is the main cleaned dataset for Task 4 and the rule/explanation pipeline.

**B. `ThreatLens_ML_Metadata.csv`**  
Contains feature-level metadata for later modeling.

Scaling is intentionally demonstrated separately after the train/test split to avoid data leakage.


In [ ]:
# 25. Save the main cleaned dataset
# Keep the original 78-column structure (77 features + original multi-class Label).
cleaned_main = df.drop(columns=["Threat_Binary"])

cleaned_path = "/content/ThreatLens_Cleaned.csv"
cleaned_main.to_csv(cleaned_path, index=False)

print("Saved:", cleaned_path)
print("Shape:", cleaned_main.shape)


In [ ]:
# 26. Create feature metadata
metadata_rows = []

for col in cleaned_main.columns:
    metadata_rows.append({
        "Feature": col,
        "Data_Type": str(cleaned_main[col].dtype),
        "Missing_Count": int(cleaned_main[col].isna().sum()),
        "Unique_Values": int(cleaned_main[col].nunique()),
        "Role": (
            "Original Target"
            if col == "Label"
            else "Binary Target"
            if False
            else "Input Feature"
        )
    })

metadata_df = pd.DataFrame(metadata_rows)
display(metadata_df.head(20))

metadata_path = "/content/ThreatLens_Feature_Metadata.csv"
metadata_df.to_csv(metadata_path, index=False)
print("Saved:", metadata_path)


## 11. Scaling demonstration for future ML

For machine learning, numerical features should be standardized when the selected algorithm benefits from scaling.

**Important:** The scaler must be fitted only on the training set. Fitting it on the entire dataset before splitting would leak information from the test set.


In [ ]:
# 27. Train/test split and leakage-safe scaling
X = df.drop(columns=["Label", "Threat_Binary"])
y = df["Threat_Binary"]

# All remaining columns are expected to be numeric in this dataset.
non_numeric_X = X.select_dtypes(exclude=np.number).columns.tolist()
print("Non-numeric model columns:", non_numeric_X)

# Convert to numeric defensively
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Fill any conversion-created missing values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training features:", X_train.shape[1])
print("Attack ratio:", round(y.mean(), 4))


In [ ]:
# 28. Verify scaling
print("Training scaled mean (first 10 features):")
print(np.round(X_train_scaled.mean(axis=0)[:10], 4))

print("\nTraining scaled std (first 10 features):")
print(np.round(X_train_scaled.std(axis=0)[:10], 4))


## 12. Final validation

Task 4 is complete when:
- the dataset is loaded and combined,
- feature types are understood,
- summary statistics are produced,
- missing/infinite values are handled,
- duplicates are removed,
- outliers are identified and treated carefully,
- visual patterns are documented,
- the cleaned dataset is saved,
- scaling is prepared without test-set leakage.


In [ ]:
# 29. Final Task 4 validation report
final_report = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns in main cleaned dataset",
        "Original Target Classes",
        "Missing Values",
        "Duplicate Rows",
        "Infinite Values",
        "Numerical Features",
        "Categorical Features"
    ],
    "Value": [
        len(cleaned_main),
        len(cleaned_main.columns),
        cleaned_main["Label"].nunique(),
        int(cleaned_main.isnull().sum().sum()),
        int(cleaned_main.duplicated().sum()),
        int(np.isinf(cleaned_main.select_dtypes(include=np.number)).sum().sum()),
        len(cleaned_main.select_dtypes(include=np.number).columns),
        len(cleaned_main.select_dtypes(exclude=np.number).columns)
    ]
})

display(final_report)


In [ ]:
# 30. Download outputs
from google.colab import files

files.download("/content/ThreatLens_Cleaned.csv")
files.download("/content/ThreatLens_Feature_Metadata.csv")


# Task 4 Conclusion

The CICIDS2017 security-flow dataset was successfully loaded, combined, explored and cleaned for ThreatLens. The analysis examined dataset structure, feature types, summary statistics, class distribution, missing and infinite values, duplicates, outliers, correlations and feature distributions. Invalid values were treated and duplicate rows were removed. Statistical outliers were identified but not blindly deleted because extreme network traffic may represent genuine attacks. The cleaned multi-class dataset was exported for subsequent ThreatLens machine-learning, deep-learning and explainability phases. A leakage-safe StandardScaler procedure was also prepared for future modeling.
